[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jiehou-lab/urban-ai/blob/main/notebooks/lab5_risk_response_dss.ipynb)

# Lab 5: Risk-Response DSS

**Duration:** ~1.25 hours
**TA lead:** Yura
**Course:** Urban AI — AI-Driven Decision Support for Real-World Urban Challenges (MSU AI-Ready Initiative)

## Learning goals
- Build a weighted-sum decision matrix over flood and heat risk criteria.
- Rank intervention alternatives and interpret a weighted score.
- Run a sensitivity sweep and see how rankings can flip as weights change.
- Produce a ranked-alternatives table with a sensitivity chart.


## Before you start: Track A vs. Track B

Every Urban AI lab has two tracks. Both produce the **same artifact**: **Ranked alternatives + sensitivity chart**.

- **Track A — No code (default).** Fill out a weighted decision matrix in a spreadsheet template. No installation, no Python required — use this track if you would rather click through a web tool.
- **Track B — Colab (this notebook).** Compute a weighted-sum score and run a sensitivity sweep on flood/heat risk criteria. You will run pre-written cells and change only the parameters marked `# ▶ CHANGE ME`. You will never need to write code from scratch.

Both tracks end with the same 4 reflection prompts (the last cell of this notebook).


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RNG = np.random.default_rng(5)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
print("Setup complete.")

## 2. Load the data: flood & heat exposure by tract

> **Synthetic-but-realistic data.** The dataset below is generated in this notebook with a fixed
> random seed so the lab runs the same way for everyone, completely offline. It is built to look and
> behave like real urban data, but it is not real. To swap in real data for your own city, instructors
> can replace the data-generation cell with a download/load from a real source such as:
- FEMA National Flood Hazard Layer / flood risk maps
- NOAA or EPA heat-vulnerability and extreme-heat datasets
- CDC/ATSDR Social Vulnerability Index (SVI) for population vulnerability indicators
- City hazard mitigation plans, which often already contain a comparable risk table


In [ ]:
n_alt = 10
alt_names = [f"Tract {chr(65 + i)}" for i in range(n_alt)]

flood_risk = RNG.uniform(0.1, 1.0, n_alt)
heat_risk = RNG.uniform(0.1, 1.0, n_alt)
population_vulnerability = RNG.uniform(0.1, 1.0, n_alt)  # elderly / low-income / no-AC share
mitigation_cost = RNG.uniform(0.2, 1.0, n_alt)  # normalized cost of intervention (higher = costlier)

decision_matrix = pd.DataFrame({
    "alternative": alt_names,
    "flood_risk": flood_risk.round(2),
    "heat_risk": heat_risk.round(2),
    "population_vulnerability": population_vulnerability.round(2),
    "mitigation_cost": mitigation_cost.round(2),
}).set_index("alternative")
decision_matrix

### Turn cost into a benefit
Flood risk, heat risk, and vulnerability are all "higher is more urgent" criteria. Cost is the opposite -- higher cost is worse -- so we invert it into a `cost_efficiency` score before combining everything on the same 0-1 scale.

In [ ]:
scored = decision_matrix.copy()
scored["cost_efficiency"] = 1 - scored["mitigation_cost"]
scored[["flood_risk", "heat_risk", "population_vulnerability", "cost_efficiency"]]

### Apply weights and rank the alternatives
The weighted-sum method is one of the most common (and most transparent) DSS techniques: each criterion gets a weight reflecting its importance, and alternatives are ranked by their weighted total.

In [ ]:
# ▶ CHANGE ME: default weights (must sum to roughly 1.0)
weights = {"flood_risk": 0.30, "heat_risk": 0.30, "population_vulnerability": 0.30, "cost_efficiency": 0.10}

scored["weighted_score"] = sum(scored[c] * w for c, w in weights.items())
ranked = scored.sort_values("weighted_score", ascending=False)
ranked[["weighted_score"] + list(weights.keys())].round(3)

In [ ]:
fig_ranked, ax = plt.subplots(figsize=(8, 5))
ranked["weighted_score"].plot(kind="bar", ax=ax, color="tab:green")
ax.set_title("Ranked Alternatives: Weighted Risk-Response Score")
ax.set_xlabel("Alternative (tract)")
ax.set_ylabel("Weighted score (higher = higher priority)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### Responsible AI check: sensitivity sweep
Decision weights are judgment calls, not facts. Here we vary the weight placed on population vulnerability and recompute the ranking each time -- watch how some tracts change rank, and even trade places, as the weight shifts.

In [ ]:
def compute_ranks(w_flood, w_heat, w_vuln, w_cost_eff):
    s = scored.copy()
    s["score"] = (s["flood_risk"] * w_flood + s["heat_risk"] * w_heat +
                  s["population_vulnerability"] * w_vuln + s["cost_efficiency"] * w_cost_eff)
    return s["score"].rank(ascending=False, method="min")


vuln_weight_range = np.arange(0.05, 0.65, 0.05)
rank_table = pd.DataFrame(index=decision_matrix.index)
for wv in vuln_weight_range:
    remaining = 1 - wv
    w_flood = w_heat = remaining * 0.4
    w_cost_eff = remaining * 0.2
    rank_table[f"{wv:.2f}"] = compute_ranks(w_flood, w_heat, wv, w_cost_eff)

fig_sensitivity, ax = plt.subplots(figsize=(10, 6))
for alt in rank_table.index:
    ax.plot(vuln_weight_range, rank_table.loc[alt], marker="o", label=alt)
ax.set_title("Sensitivity Sweep: Rank vs. Population-Vulnerability Weight")
ax.set_xlabel("Weight on population vulnerability")
ax.set_ylabel("Rank (1 = highest priority)")
ax.invert_yaxis()
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Notice how several lines cross: an alternative that looks top-priority at one weight setting can drop several places at another. **A ranked list from a single weight choice is not a fact -- it is one judgment call among several defensible ones.**

## Experiment

Try changing the parameters marked `# ▶ CHANGE ME` in the next cell(s) and re-run. Specifically, try:

1. Change the `weights` dictionary above to put more emphasis on `cost_efficiency` -- does the #1-ranked alternative change?
2. Widen `vuln_weight_range` (e.g., up to 0.8) -- do more rank crossovers appear?
3. Set `population_vulnerability` weight to 0 entirely -- which alternatives lose priority?


## Artifact: ranked alternatives + sensitivity chart

In [ ]:
ranked_display = ranked[["weighted_score", "flood_risk", "heat_risk",
                         "population_vulnerability", "mitigation_cost"]].round(3)
print(ranked_display)

ranked_display.to_csv("lab5_ranked_alternatives.csv")
fig_sensitivity.savefig("lab5_sensitivity_chart.png", dpi=150, bbox_inches="tight")
print("\nSaved artifacts: lab5_ranked_alternatives.csv, lab5_sensitivity_chart.png")

## Reflect (answer in your own words — this is part of your mini-task)

1. What did the tool assume?
2. Who is missing from this data?
3. What would change your recommendation?
4. What must a human verify before this is used?
